In [14]:
import pandas as pd
import numpy as np
from sklearn import tree
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import os
import datetime
from tqdm import tqdm

In [15]:
df = pd.read_csv("data/youtube_telegram_cross.csv")
df_train = pd.read_csv("sample_yt_te_cross_with_topics.csv")

print(len(df))
print(len(df_train))

673587
67357


In [16]:
# Corrige alinhamento de índices
df_embeddings = df[['video_id']].copy()
df_embeddings['embedding_index'] = np.arange(len(df))
df_train_merged = df_train.merge(df_embeddings, on='video_id', how='left')

missing = df_train_merged['embedding_index'].isna().sum()
print(f"Linhas sem embedding correspondente: {missing}")

Linhas sem embedding correspondente: 0


In [17]:
train_idx = df_train_merged['embedding_index'].dropna().astype(int).values
y = df_train_merged["interest"].values

embeddings = np.load("classification_models/embeddings.npy")
embeddings_reduced = np.load("classification_models/embeddings_reduced.npy")

X = embeddings[train_idx]
X_reduced = embeddings_reduced[train_idx]

In [18]:
models = [
    ("RF (reduced)", RandomForestClassifier(random_state=42),
     {'n_estimators': [100, 200], 'max_depth': [None, 10], 'min_samples_split': [2, 5],  'class_weight': [None, 'balanced']}),
]

In [19]:
def run_model(name, model, param_grid, X, y, cv_splits=5):
    print(f"\n Rodando modelo: {name}")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    strat_kfold = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

    clf = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=strat_kfold,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1
    )

    clf.fit(X_scaled, y)
    best_model = clf.best_estimator_
    print(f"Melhores hiperparâmetros encontrados: {clf.best_params_}")

    print(f"=== {name} ===\n")
    print(f"Melhores parâmetros: {clf.best_params_}\n\n")

    reports = []
    for fold, (train_idx, test_idx) in enumerate(strat_kfold.split(X_scaled, y), 1):
        y_pred = best_model.fit(X_scaled[train_idx], y[train_idx]).predict(X_scaled[test_idx])
        report = classification_report(y[test_idx], y_pred, digits=3, output_dict=True)
        reports.append(report)

        print(f"\n--- Fold {fold} ---\n")
        print(classification_report(y[test_idx], y_pred, digits=3))
        print("\n")

    avg_f1_macro = np.mean([r['macro avg']['f1-score'] for r in reports])
    std_f1_macro = np.std([r['macro avg']['f1-score'] for r in reports])
    print(f"\nMédia F1 Macro: {avg_f1_macro:.3f} ± {std_f1_macro:.3f}\n")

    print(f"✅ Finalizado: {name} | Média F1 Macro: {avg_f1_macro:.3f}")
    return best_model

In [20]:
for name, model, params in tqdm(models, desc="Treinando modelos", ncols=100):
    X_input = X_reduced if "reduced" in name else X
    best_model = run_model(name, model, params, X_input, y)


Treinando modelos:   0%|                                                      | 0/1 [00:00<?, ?it/s]


 Rodando modelo: RF (reduced)
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Melhores hiperparâmetros encontrados: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
=== RF (reduced) ===

Melhores parâmetros: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}



--- Fold 1 ---

              precision    recall  f1-score   support

           0      0.779     0.759     0.769      5373
           1      0.843     0.858     0.850      8099

    accuracy                          0.818     13472
   macro avg      0.811     0.808     0.810     13472
weighted avg      0.817     0.818     0.818     13472




--- Fold 2 ---

              precision    recall  f1-score   support

           0      0.788     0.774     0.781      5374
           1      0.852     0.862     0.857      8098

    accuracy                          0.827     13472
   macro avg      0.820     0.818     0.819     13472
weig

Treinando modelos: 100%|█████████████████████████████████████████████| 1/1 [04:53<00:00, 293.35s/it]


--- Fold 5 ---

              precision    recall  f1-score   support

           0      0.773     0.761     0.767      5373
           1      0.843     0.851     0.847      8098

    accuracy                          0.815     13471
   macro avg      0.808     0.806     0.807     13471
weighted avg      0.815     0.815     0.815     13471




Média F1 Macro: 0.812 ± 0.005

✅ Finalizado: RF (reduced) | Média F1 Macro: 0.812


In [21]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_reduced)

In [22]:
# embeddings reduzidos para TODOS os vídeos
all_embeddings_reduced = embeddings_reduced

# aplica o mesmo scaler usado no treino
all_embeddings_scaled = scaler.transform(all_embeddings_reduced)

# gera predições: 0 = sem interesse, 1 = interesse
pred_all = best_model.predict(all_embeddings_scaled)
prob_all = best_model.predict_proba(all_embeddings_scaled)[:, 1]  # probabilidade de interesse


In [23]:
df['interest_pred'] = pred_all
df['interest_prob'] = prob_all

In [24]:
df.to_csv("data/youtube_telegram_predicted_interest.csv", index=False)

In [ ]:
display(df)

,video_id,text,channel_title,published_at,view_count,like_count,comment_count,occurrences,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,interest_pred,interest_prob
0,jEKzQV5oajY,travel migrant scrap minute,Joe Marsh,2024-04-10T15:05:37Z,1057.0,152.0,32.0,"[{'id': 1416, 'folder': 'channel_1556142220', ...",0.000659,0.000117,0.000183,0.000114,0.000178,0.000138,0,0.067799
1,xrGGce8cmx8,spiritual warfare charge commit unto thee timo...,WWURD,2023-10-18T14:39:18Z,65.0,6.0,1.0,"[{'id': 60799, 'folder': 'channel_1466271872',...",0.295000,0.001137,0.003694,0.012858,0.003023,0.006275,1,0.739668
2,uaozGpSc4nc,putin layer february tucker carlson stand onio...,The Mosaic Ark,2024-02-22T07:30:05Z,330.0,8.0,6.0,"[{'id': 40486, 'folder': 'channel_1235978663',...",0.001735,0.000102,0.000245,0.000142,0.000204,0.000156,0,0.166233
3,A59ftbhsQUE,GALACTIC ALLIANCE MESSAGE NEWS REMAIN ALERT PR...,GALACTIC ALLIANCE,2024-10-27T04:30:04Z,806.0,136.0,18.0,"[{'id': 449843, 'folder': 'channel_1571505334'...",0.002710,0.000104,0.000254,0.000185,0.000239,0.000192,1,0.811967
4,AKU0RokegSo,angeles rain studio city evacuate mudslide atm...,FOX 11 Los Angeles,2024-02-06T01:45:29Z,60863.0,406.0,152.0,"[{'id': 104116, 'folder': 'channel_1438734111'...",0.001515,0.000100,0.000217,0.000126,0.000203,0.000143,1,0.779615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
673582,ABpufr8inCY,eruption kilauea volcano earthquake activity i...,TheEarthMaster,2024-09-17T19:15:25Z,10563.0,712.0,38.0,"[{'id': 30195, 'folder': 'channel_1177617529',...",0.003324,0.000098,0.000358,0.000123,0.000247,0.000152,1,0.991994
673583,5VWWCeGrr-w,eslteacher funny kidsvideo play easy makeup le...,Game fanny,2024-05-08T22:00:03Z,2.0,0.0,0.0,"[{'id': 1531863, 'folder': 'channel_1391678616...",0.004620,0.000084,0.000295,0.000108,0.000313,0.000175,0,0.379915
673584,RxbPFQp0SHs,unbelievable ending play bgmi bgmi gameplay su...,𝐒∆𝐈𝐘𝐀𝐍 • 127K views • 6 hours ago,2023-12-29T23:30:11Z,82.0,4.0,0.0,"[{'id': 197485, 'folder': 'channel_1215588509'...",0.023877,0.000158,0.002297,0.000265,0.001188,0.000347,0,0.076602
673585,GEhEUy85PsY,meet balkan,Icariaball,2021-04-22T16:25:46Z,7617763.0,210152.0,14042.0,"[{'id': 7719, 'folder': 'channel_1753496417', ...",0.007485,0.000123,0.000266,0.000247,0.000321,0.000520,0,0.302432


In [26]:
df["interest_pred"].value_counts()

interest_pred
1    410013
0    263574
Name: count, dtype: int64